# 🚀 ColabTube Uploader
Công cụ đa năng tải video từ **mọi nguồn** và Upload trực tiếp lên YouTube siêu tốc.

**Nguồn hỗ trợ:** Google Drive, YouTube, OK.ru, VK, Facebook, TikTok, Bilibili (.com & .tv), **Link Torrent / Magnet**, và link trực tiếp (.mp4).

**Hướng dẫn:** Chạy từng ô từ trên xuống dưới bằng nút ▶️ bên trái.


In [ ]:
#@title 📦 Cài đặt thư viện nền tảng (Chạy 1 lần)
!pip install --upgrade -q google-api-python-client google-auth-oauthlib google-auth-httplib2 httplib2
!pip install --force-reinstall -q yt-dlp
!apt-get update -y -q && apt-get install -y -q ffmpeg aria2


In [ ]:
#@title 💾 Kết nối Google Drive (Sao lưu cấu hình tự động)
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

BACKUP_DIR = '/content/drive/MyDrive/ColabTube_Backup'
os.makedirs(BACKUP_DIR, exist_ok=True)

# Khôi phục file cấu hình từ Drive (nếu có)
restored = []
for fname in ['client_secrets.json', 'youtube_token.json']:
    backup_path = os.path.join(BACKUP_DIR, fname)
    if os.path.exists(backup_path) and not os.path.exists(fname):
        shutil.copy2(backup_path, fname)
        restored.append(fname)

if restored:
    print(f"✅ Đã khôi phục từ Drive: {', '.join(restored)}")
    print("👉 Bạn có thể BỎ QUA bước tải client_secrets.json bên dưới!")
else:
    print("✅ Đã kết nối Google Drive thành công.")
    print("📁 Thư mục sao lưu: Drive/ColabTube_Backup/")


In [ ]:
#@title 🔑 Tải lên file chứng chỉ client_secrets.json (Chạy 1 lần)
from google.colab import files
import os, shutil

BACKUP_DIR = '/content/drive/MyDrive/ColabTube_Backup'

if os.path.exists('client_secrets.json'):
    print("✅ File client_secrets.json đã tồn tại. Bỏ qua bước này.")
else:
    print("Vui lòng tải lên file client_secrets.json của bạn (Lấy từ Google Cloud Console)")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn != 'client_secrets.json':
            os.rename(fn, 'client_secrets.json')
    print("✅ Đã tải lên thành công file cấu hình API.")

# Sao lưu vào Drive
if os.path.exists(BACKUP_DIR) and os.path.exists('client_secrets.json'):
    shutil.copy2('client_secrets.json', os.path.join(BACKUP_DIR, 'client_secrets.json'))
    print("💾 Đã sao lưu client_secrets.json vào Drive/ColabTube_Backup/")


In [ ]:
#@title ⚙️ Cấu hình Video

#@markdown Dán đường link Video vào đây. Hỗ trợ: Google Drive, YouTube, OK.ru, VK, Facebook, TikTok, Bilibili, **Link Torrent / Magnet** và link trực tiếp.
VIDEO_LINK = "" #@param {type:"string"}
TRANG_THAI_VIDEO = "Riêng tư (private)" #@param ["Công khai (public)", "Riêng tư (private)", "Không công khai (unlisted)"]
CHIA_NHO_VIDEO = "Kh\u00f4ng c\u1eaft" #@param ["Kh\u00f4ng c\u1eaft", "C\u1eaft l\u00e0m 2 ph\u1ea7n", "C\u1eaft l\u00e0m 3 ph\u1ea7n", "C\u1eaft l\u00e0m 5 ph\u1ea7n", "C\u1eaft l\u00e0m 7 ph\u1ea7n"]

#@markdown **Lưu ý:** Phần chữ ký dưới đây sẽ tự động chèn vào nội dung mô tả của Video trên YouTube.
MO_TA_VIDEO = """📌Contact: https://beacons.ai/huyvu2512
------------------------------------------------------------------------------------------------------------------------
📌Mọi Vấn Đề Về Bản Quyền Hình Ảnh Và Video Hãy Liên Hệ Tôi Qua Email: quanghuy25121995@gmail.com 
📌Any Issues About Image And Video Copyright Please Contact Me By Email: quanghuy25121995@gmail.com"""


In [ ]:
#@title 🚀 BẤM NÚT NÀY ĐỂ BẮT ĐẦU TẢI VÀ UPLOAD YOUTUBE
import os
import re
import json
import io
import time
from google.colab import auth
from google_auth_oauthlib.flow import Flow
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from googleapiclient.errors import HttpError, ResumableUploadError
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request

def format_time(seconds):
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    if h > 0: return f"{h}h {m}m {s}s"
    elif m > 0: return f"{m}m {s}s"
    return f"{s}s"

def format_size(bytes_val):
    if not bytes_val: return "N/A"
    bytes_val = float(bytes_val)
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if bytes_val < 1024.0:
            return f"{bytes_val:.1f} {unit}"
        bytes_val /= 1024.0
    return f"{bytes_val:.1f} PB"

# Bỏ qua lỗi bắt buộc HTTPS khi dùng localhost (Fix InsecureTransportError)
os.environ['OAUTHLIB_INSECURE_TRANSPORT'] = '1'

token_file = 'youtube_token.json'

# 1. Xác thực YouTube API theo Kênh
if not os.path.exists('client_secrets.json'):
    raise FileNotFoundError("❌ Không tìm thấy file client_secrets.json! Vui lòng chạy bước tải file lên ở trên.")

youtube_credentials = None
if os.path.exists(token_file):
    youtube_credentials = Credentials.from_authorized_user_file(token_file, ['https://www.googleapis.com/auth/youtube.upload'])

if not youtube_credentials or not youtube_credentials.valid:
    if youtube_credentials and youtube_credentials.expired and youtube_credentials.refresh_token:
        youtube_credentials.refresh(Request())
    else:
        with open('client_secrets.json', 'r') as f:
            client_config = json.load(f)
        
        redirect_uri = 'http://localhost:8080/'
        if 'installed' in client_config and 'redirect_uris' in client_config['installed']:
            redirect_uri = client_config['installed']['redirect_uris'][0]
        elif 'web' in client_config and 'redirect_uris' in client_config['web']:
            redirect_uri = client_config['web']['redirect_uris'][0]

        scopes = ['https://www.googleapis.com/auth/youtube.upload']
        flow = Flow.from_client_secrets_file('client_secrets.json', scopes=scopes, redirect_uri=redirect_uri)
        
        # ÉP GOOGLE HIỆN BẢNG CHỌN KÊNH bằng tham số prompt='consent select_account'
        auth_url, _ = flow.authorization_url(prompt='consent select_account')
        
        print(f'\n=== 🔑 HƯỚNG DẪN XÁC THỰC KÊNH YOUTUBE ===')
        print('1. Nhấn vào link sau để mở trang đăng nhập Google:')
        print(auth_url)
        print('\n2. CHỌN ĐÚNG KÊNH YOUTUBE BẠN MUỐN UP VIDEO và cấp quyền.')
        print('3. Trình duyệt sẽ báo lỗi (ví dụ: không truy cập được localhost).')
        print('4. Copy ĐƯỜNG DẪN (URL) của trang lỗi đó dán vào ô bên dưới.')
        print('====================================================\n')
        
        redirect_response = input('Dán toàn bộ URL vừa copy vào đây và nhấn Enter: ').strip()
        if redirect_response.startswith('http://'):
            redirect_response = redirect_response.replace('http://', 'https://', 1)
        
        import urllib.parse as urlparse
        from urllib.parse import parse_qs
        parsed = urlparse.urlparse(redirect_response)
        code = parse_qs(parsed.query).get('code', [None])[0]
        if code:
            flow.fetch_token(code=code)
        else:
            flow.fetch_token(authorization_response=redirect_response)
        youtube_credentials = flow.credentials
        
    # Lưu lại token để dùng cho các video sau trên kênh này
    with open(token_file, 'w') as f:
        f.write(youtube_credentials.to_json())

# Sao lưu token vào Drive
BACKUP_DIR = '/content/drive/MyDrive/ColabTube_Backup'
if os.path.exists(BACKUP_DIR):
    import shutil
    shutil.copy2(token_file, os.path.join(BACKUP_DIR, token_file))

print(f"✅ Đã tải thông tin xác thực cho kênh.")

# 2. Xử lý link Google Drive và tải file
import yt_dlp
import glob

# --- CƠ CHẾ KHÔI PHỤC (RESUME) KHI CHẠY LẠI ---
STATE_FILE = '/content/upload_state.json'
state = {}
if os.path.exists(STATE_FILE):
    try:
        with open(STATE_FILE, 'r', encoding='utf-8') as sf:
            state = json.load(sf)
    except:
        pass

# Nếu link mới khác link cũ đang chạy dở, dọn dẹp và reset state
if state.get('video_link') != VIDEO_LINK:
    print("🧹 Phát hiện link mới, đang dọn dẹp các file cũ...")
    # Dọn dẹp các file tải cũ
    for f in glob.glob('/content/downloaded_video*'):
        try: os.remove(f)
        except: pass
    for f in glob.glob('/content/*_Phan_*'):
        try: os.remove(f)
        except: pass
    if os.path.exists(STATE_FILE):
        try: os.remove(STATE_FILE)
        except: pass
    state = {
        "video_link": VIDEO_LINK,
        "step": "start",
        "downloaded_file": "",
        "upload_files": []
    }
else:
    print("🔄 Phát hiện tiến trình cũ chạy dở. Đang khôi phục để chạy tiếp...")

def save_state():
    with open(STATE_FILE, 'w', encoding='utf-8') as sf:
        json.dump(state, sf, indent=2, ensure_ascii=False)

def is_drive_link(url):
    return 'drive.google.com' in url or 'docs.google.com' in url

def get_drive_id(url):
    match = re.search(r'/d/([a-zA-Z0-9_-]+)', url)
    if match: return match.group(1)
    match = re.search(r'id=([a-zA-Z0-9_-]+)', url)
    if match: return match.group(1)
    return url.strip()

if not VIDEO_LINK.strip():
    raise ValueError("❌ BẠN CHƯA NHẬP LINK! Vui lòng dán link Video vào ô cấu hình bên trên trước khi chạy.")

file_name = 'video.mp4'
FILE_PATH = state.get('downloaded_file', '')

# Chỉ tải video nếu chưa tải xong trước đó
if state.get('step') == 'start' or not FILE_PATH or not os.path.exists(FILE_PATH):
    if is_drive_link(VIDEO_LINK):
        # --- TẢI TỪ GOOGLE DRIVE ---
        file_id = get_drive_id(VIDEO_LINK)
        print("⏳ Đang xử lý yêu cầu quyền truy cập Google Drive...")
        auth.authenticate_user()
        drive_service = build('drive', 'v3')
        try:
            file_info = drive_service.files().get(fileId=file_id, fields='name, size', supportsAllDrives=True).execute()
        except Exception as e:
            raise Exception(f"❌ Không thể truy cập file Drive này! Chi tiết lỗi: {e}")
        file_name = file_info.get('name', 'video.mp4')
        total_size_bytes = int(file_info.get('size', 0)) if file_info.get('size') else 0
        size_str = format_size(total_size_bytes) if total_size_bytes else "N/A"
        FILE_PATH = f"/content/{file_name}"
        if not os.path.exists(FILE_PATH):
            print(f"\n🎬 File: {file_name} ({size_str})")
            request = drive_service.files().get_media(fileId=file_id, acknowledgeAbuse=True, supportsAllDrives=True)
            fh = io.FileIO(FILE_PATH, 'wb')
            downloader = MediaIoBaseDownload(fh, request, chunksize=1024*1024*100)
            done = False
            start_time = time.time()
            while done is False:
                status, done = downloader.next_chunk()
                if status:
                    progress = status.progress()
                    elapsed = time.time() - start_time
                    if progress > 0 and elapsed > 0:
                        speed = status.resumable_progress / elapsed / (1024 * 1024)
                        eta = (elapsed / progress) - elapsed
                        print(f"\r⬇️ Đang tải: {int(progress * 100)}% | Tốc độ: {speed:.1f} MB/s | Còn lại: {format_time(eta)}      ", end="")
            print("\n✅ Đã kéo xong video!")
        else:
            print(f"✅ Video '{file_name}' đã có sẵn trong Colab.")
    else:
        # --- TẢI TỪ CÁC NGUỒN KHÁC BẰNG YT-DLP ---
        print("🔥 BẮT ĐẦU KÉO VIDEO TỪ NGUỒN BÊN NGOÀI...")
        is_torrent = VIDEO_LINK.startswith('magnet:?') or VIDEO_LINK.endswith('.torrent') or '.torrent?' in VIDEO_LINK
        if is_torrent:
            print("📦 Đang chuẩn bị tải Torrent...")
            if os.system('which aria2c > /dev/null 2>&1') != 0:
                print("📦 Đang cài đặt công cụ tải torrent (aria2)... Vui lòng chờ...")
                os.system('apt-get update -y -q && apt-get install -y -q aria2')
            
            import shutil
            import subprocess
            import glob
            torrent_dir = '/content/torrent_download'
            if not os.path.exists(torrent_dir):
                os.makedirs(torrent_dir, exist_ok=True)
            
            print("⬇️ Đang tải video từ Torrent...")
            trackers = ""
            try:
                import urllib.request
                req = urllib.request.Request('https://cf.trackerslist.com/best.txt', headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=5) as response:
                    trackers_list = response.read().decode('utf-8').strip().split('\n')
                    trackers = ','.join([t.strip() for t in trackers_list if t.strip()])
            except Exception:
                pass
            
            cmd = ['aria2c', '--seed-time=0', '--summary-interval=5', '--bt-enable-lpd=true', '--enable-dht=true', '--enable-dht6=true', '--max-connection-per-server=16', '--split=16', '-d', torrent_dir]
            if trackers:
                cmd.append(f'--bt-tracker={trackers}')
            cmd.append(VIDEO_LINK)
            process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            import re
            aria_progress_pattern = re.compile(
                r'\[#(?:\w+)\s+(?P<downloaded>[^\s/]+)/(?P<total>[^\s\(\]]+)(?:\((?P<percent>\d+)%\))?.*?(?:CN:(?P<cn>\d+))?.*?(?:SD:(?P<sd>\d+))?.*?(?:DL:(?P<speed>[^\s\]]+))?(?:\s+ETA:(?P<eta>[^\s\]]+))?\]'
            )
            torrent_name = "Đang chuẩn bị..."
            printed_torrent_info = False
            for pline in process.stdout:
                pline_str = pline.strip()
                if 'FILE:' in pline_str:
                    parts = pline_str.split('FILE:', 1)
                    file_path_info = parts[1].strip()
                    if file_path_info.startswith('[MEMORY][METADATA]'):
                        torrent_name = file_path_info.replace('[MEMORY][METADATA]', '').replace('+', ' ').strip()
                    else:
                        torrent_name = os.path.basename(file_path_info)
                    continue
                
                if '[MEMORY][METADATA]' in pline_str or 'Download complete:' in pline_str:
                    continue
                
                match = aria_progress_pattern.search(pline_str)
                if match:
                    percent_val = match.group('percent')
                    total_val = match.group('total')
                    cn_val = match.group('cn')
                    sd_val = match.group('sd')
                    speed_val = match.group('speed')
                    eta_val = match.group('eta')
                    
                    cn = int(cn_val) if cn_val else 0
                    sd = int(sd_val) if sd_val else 0
                    
                    total_size = total_val.replace('MiB', ' MB').replace('KiB', ' KB').replace('GiB', ' GB') if total_val else "N/A"
                    
                    if percent_val is None or percent_val == '0' or total_val == '0B':
                        if cn == 0:
                            print(f'\r⏳ Đang tìm nguồn phát (Seeds)... (Chưa kết nối được máy nào)      ', end='', flush=True)
                        else:
                            print(f'\r⏳ Đang tải thông tin Torrent (Metadata)... (Đã kết nối {cn} máy)      ', end='', flush=True)
                    else:
                        if not printed_torrent_info and torrent_name != "Đang chuẩn bị...":
                            print(f'\n🎬 File: {torrent_name} ({total_size})')
                            printed_torrent_info = True
                        
                        percent = percent_val + '%'
                        speed = speed_val.replace('MiB', ' MB/s').replace('KiB', ' KB/s').replace('GiB', ' GB/s') if speed_val else '0 KB/s'
                        if eta_val:
                            eta = eta_val.replace('h', 'h ').replace('m', 'm ').strip()
                        else:
                            if sd == 0 and (not speed_val or '0B' in speed_val):
                                eta = 'Đang chờ nguồn phát (Seeds: 0)...'
                            else:
                                eta = 'Đang tính...'
                        print(f'\r⬇️ Đang tải: {percent} | Tốc độ: {speed} | Còn lại: {eta}      ', end='', flush=True)
            
            process.wait()
            if process.returncode != 0:
                raise Exception("❌ Tải torrent thất bại! Vui lòng kiểm tra lại link hoặc seed của torrent.")
            
            video_extensions = ('.mp4', '.mkv', '.avi', '.ts', '.mov', '.flv', '.webm', '.m4v')
            video_files = []
            for root, dirs, files_in_dir in os.walk(torrent_dir):
                for f in files_in_dir:
                    if f.lower().endswith(video_extensions):
                        full_path = os.path.join(root, f)
                        video_files.append((full_path, os.path.getsize(full_path)))

            if not video_files:
                raise Exception("❌ Không tìm thấy file video nào (.mp4, .mkv, ...) trong torrent tải về!")
            
            video_files.sort(key=lambda x: x[1], reverse=True)
            FILE_PATH = video_files[0][0]
            file_name = os.path.basename(FILE_PATH)
            print(f"\n✅ Đã tải xong Torrent! File: {file_name} ({video_files[0][1] / (1024*1024):.1f} MB)")
        else:
            class MyLogger:
                def debug(self, msg):
                    if 'Extracting URL' in msg:
                        print(f'🔍 Đang trích xuất link video...')
                    elif 'Downloading webpage' in msg or 'Downloading desktop webpage' in msg:
                        print(f'🌐 Đang tải thông tin trang web...')
                    elif 'Downloading m3u8 information' in msg:
                        print(f'📄 Đang tải luồng video...')
                def info(self, msg):
                    pass
                def warning(self, msg):
                    print(f'⚠️ Cảnh báo: {msg}')
                def error(self, msg):
                    print(f'❌ Lỗi tải video: {msg}')
        
            def my_hook(d):
                if d['status'] == 'downloading':
                    percent = d.get('_percent_str', '0%').strip()
                    speed = d.get('_speed_str', '0B/s').strip()
                    eta = d.get('_eta_str', 'N/A').strip()
                    percent = re.sub(r'\x1b\[[0-9;]*m', '', percent).replace(' ', '')
                    speed = re.sub(r'\x1b\[[0-9;]*m', '', speed).replace('MiB/s', 'MB/s').replace('GiB/s', 'GB/s').replace('KiB/s', 'KB/s').replace(' ', '')
                    eta = re.sub(r'\x1b\[[0-9;]*m', '', eta)
                    if ':' in eta:
                        parts = eta.split(':')
                        if len(parts) == 3:
                            eta = f'{int(parts[0])}h {int(parts[1])}m {int(parts[2])}s'
                        elif len(parts) == 2:
                            eta = f'{int(parts[0])}m {int(parts[1])}s'
                    
                    info = d.get('info_dict', {})
                    title = info.get('title', 'Video')
                    total_bytes = d.get('total_bytes') or d.get('total_bytes_estimate') or info.get('filesize') or info.get('filesize_approx')
                    size_str = format_size(total_bytes) if total_bytes else "N/A"
                    
                    if not my_hook.printed_info and title != 'Video':
                        print(f'\n🎬 File: {title} ({size_str})')
                        my_hook.printed_info = True
                    
                    print(f'\r⬇️ Đang tải: {percent} | Tốc độ: {speed} | Còn lại: {eta}      ', end='')
                elif d['status'] == 'finished':
                    print('\n🔄 Đã tải xong phân mảnh. Đang xử lý hoặc chờ gộp file...')
            
            my_hook.printed_info = False
            ydl_opts = {
                'outtmpl': '/content/downloaded_video_%(id)s.%(ext)s',
                'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
                'merge_output_format': 'mp4',
                'quiet': True,
                'no_warnings': True,
                'logger': MyLogger(),
                'progress_hooks': [my_hook],
                'retries': 10,
                'fragment_retries': 10,
                'extractor_retries': 5,
                'file_access_retries': 5,
                'retry_sleep_functions': {'extractor': lambda n: 2 * n, 'http': lambda n: 2 * n, 'fragment': lambda n: 1},
            }
        
            if 'bilibili.com' in VIDEO_LINK:
                ydl_opts['http_headers'] = {'Referer': 'https://www.bilibili.com/'}
            elif 'bilibili.tv' in VIDEO_LINK:
                ydl_opts['http_headers'] = {'Referer': 'https://www.bilibili.tv/'}
        
            MAX_RETRIES = 3
            for attempt in range(1, MAX_RETRIES + 1):
                try:
                    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                        info_dict = ydl.extract_info(VIDEO_LINK, download=True)
                        file_name = info_dict.get('title', 'video')
                        downloaded_files = glob.glob('/content/downloaded_video_*')
                        if downloaded_files:
                            FILE_PATH = downloaded_files[0]
                        else:
                            raise Exception("❌ Không thể tải được video từ link này!")
                    print("\n✅ Đã tải xong video!")
                    break
                except Exception as e:
                    if attempt < MAX_RETRIES:
                        wait = attempt * 3
                        print(f'\n⚠️ Lỗi lần {attempt}/{MAX_RETRIES}: {str(e)[:100]}')
                        print(f'🔄 Tự động thử lại sau {wait} giây...')
                        time.sleep(wait)
                        for f in glob.glob('/content/downloaded_video*'):
                            if not f.endswith('.part') and not f.endswith('.ytdl'):
                                try: os.remove(f)
                                except: pass
                    else:
                        raise Exception(f"❌ Đã thử {MAX_RETRIES} lần nhưng không thể tải video!\n👉 Chi tiết lỗi: {e}")

    state['downloaded_file'] = FILE_PATH
    state['step'] = 'downloaded'
    save_state()
else:
    print(f"✅ Video đã tải sẵn trước đó: {os.path.basename(FILE_PATH)}")


# 2.5. TỐI ƯU ÂM THANH (Chuẩn hóa âm lượng cho YouTube)
import subprocess

def normalize_audio(input_path):
    """
    Chuẩn hóa âm thanh video theo tiêu chuẩn YouTube (-14 LUFS).
    - Tự động chuyển âm thanh 5.1/7.1 về Stereo (giữ rõ giọng nói).
    - Chuẩn hóa âm lượng bằng bộ lọc loudnorm 2 lượt (2-pass) để đạt chất lượng cao nhất.
    - Giữ nguyên luồng hình ảnh (không re-encode video).
    """
    print("\n🔊 ĐANG TỐI ƯU ÂM THANH CHO YOUTUBE...")
    
    # Kiểm tra thông tin âm thanh hiện tại
    probe_cmd = f'ffprobe -v error -select_streams a:0 -show_entries stream=channels,codec_name -of json "{input_path}"'
    try:
        probe_result = subprocess.check_output(probe_cmd, shell=True).decode('utf-8')
        probe_data = json.loads(probe_result)
        streams = probe_data.get('streams', [])
        if not streams:
            print("⚠️ Không tìm thấy luồng âm thanh, bỏ qua tối ưu.")
            return input_path
        
        channels = streams[0].get('channels', 2)
        codec = streams[0].get('codec_name', 'unknown')
        
        if channels > 2:
            print(f"   📢 Phát hiện âm thanh {channels} kênh ({codec}). Sẽ chuyển về Stereo (2 kênh).")
        else:
            print(f"   📢 Âm thanh hiện tại: {channels} kênh ({codec}).")
    except Exception as e:
        print(f"⚠️ Không đọc được thông tin âm thanh: {e}. Tiếp tục tối ưu...")
        channels = 2

    # LƯỢT 1: Phân tích âm lượng hiện tại (chỉ đo, không ghi file)
    print("   📊 Lượt 1/2: Đang phân tích mức âm lượng hiện tại...")
    analyze_filter = 'loudnorm=I=-14:TP=-1.5:LRA=11:print_format=json'
    if channels > 2:
        # Downmix 5.1/7.1 sang Stereo trước khi phân tích, giữ rõ kênh Center (giọng nói)
        analyze_filter = f'pan=stereo|FL=0.5*FC+0.707*FL+0.707*BL+0.5*LFE|FR=0.5*FC+0.707*FR+0.707*BR+0.5*LFE,{analyze_filter}'
    
    pass1_cmd = f'ffmpeg -i "{input_path}" -af "{analyze_filter}" -f null -'
    try:
        pass1_result = subprocess.run(pass1_cmd, shell=True, capture_output=True, text=True, timeout=600)
        stderr_output = pass1_result.stderr
        
        # Trích xuất JSON kết quả loudnorm từ stderr
        json_start = stderr_output.rfind('{')
        json_end = stderr_output.rfind('}') + 1
        if json_start == -1 or json_end == 0:
            print("⚠️ Không trích xuất được dữ liệu phân tích âm thanh. Bỏ qua tối ưu.")
            return input_path
        
        loudnorm_stats = json.loads(stderr_output[json_start:json_end])
        
        input_i = loudnorm_stats.get('input_i', '-24')
        input_tp = loudnorm_stats.get('input_tp', '-1')
        input_lra = loudnorm_stats.get('input_lra', '7')
        input_thresh = loudnorm_stats.get('input_thresh', '-34')
        
        print(f"   📈 Kết quả phân tích: Âm lượng trung bình = {input_i} LUFS | Peak = {input_tp} dBTP")
        
        # Kiểm tra nếu âm lượng đã gần chuẩn thì bỏ qua
        try:
            lufs_val = float(input_i)
            if -15.5 <= lufs_val <= -12.5:
                print(f"   ✅ Âm lượng đã đạt chuẩn YouTube (-14 LUFS ± 1.5). Không cần chỉnh.")
                if channels <= 2:
                    return input_path
                else:
                    print(f"   ℹ️ Tuy nhiên vẫn cần chuyển {channels} kênh về Stereo.")
        except:
            pass
        
    except subprocess.TimeoutExpired:
        print("⚠️ Phân tích âm thanh quá lâu (>10 phút). Bỏ qua tối ưu.")
        return input_path
    except Exception as e:
        print(f"⚠️ Lỗi phân tích âm thanh: {e}. Bỏ qua tối ưu.")
        return input_path
    
    # LƯỢT 2: Áp dụng chuẩn hóa chính xác dựa trên kết quả phân tích
    print("   🔧 Lượt 2/2: Đang chuẩn hóa âm lượng và xuất file mới...")
    
    base, ext = os.path.splitext(input_path)
    output_path = f"{base}_normalized{ext}"
    
    norm_filter = f'loudnorm=I=-14:TP=-1.5:LRA=11:measured_I={input_i}:measured_TP={input_tp}:measured_LRA={input_lra}:measured_thresh={input_thresh}:linear=true'
    if channels > 2:
        norm_filter = f'pan=stereo|FL=0.5*FC+0.707*FL+0.707*BL+0.5*LFE|FR=0.5*FC+0.707*FR+0.707*BR+0.5*LFE,{norm_filter}'
    
    pass2_cmd = f'ffmpeg -i "{input_path}" -c:v copy -af "{norm_filter}" -ac 2 -c:a aac -b:a 192k "{output_path}" -y -v warning -stats'
    
    try:
        process = subprocess.Popen(pass2_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            line = line.strip()
            if line.startswith('size=') or 'time=' in line:
                # Trích xuất thời gian đã xử lý
                time_match = __import__('re').search(r'time=(\S+)', line)
                if time_match:
                    print(f"\r   ⏱️ Đang xử lý: {time_match.group(1)}      ", end='', flush=True)
        process.wait()
        
        if process.returncode != 0:
            print(f"\n⚠️ Lỗi chuẩn hóa âm thanh (mã lỗi: {process.returncode}). Sử dụng file gốc.")
            if os.path.exists(output_path):
                os.remove(output_path)
            return input_path
        
        if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
            original_size = os.path.getsize(input_path)
            new_size = os.path.getsize(output_path)
            print(f"\n   ✅ Hoàn tất tối ưu âm thanh!")
            print(f"   📦 Dung lượng: {original_size / (1024*1024):.1f} MB → {new_size / (1024*1024):.1f} MB")
            
            # Thay thế file gốc bằng file đã tối ưu
            os.remove(input_path)
            os.rename(output_path, input_path)
            return input_path
        else:
            print("\n⚠️ File đầu ra không hợp lệ. Sử dụng file gốc.")
            if os.path.exists(output_path):
                os.remove(output_path)
            return input_path
            
    except Exception as e:
        print(f"\n⚠️ Lỗi xử lý âm thanh: {e}. Sử dụng file gốc.")
        if os.path.exists(output_path):
            try: os.remove(output_path)
            except: pass
        return input_path

# Áp dụng tối ưu âm thanh cho file đã tải về
if state.get('step') == 'downloaded' or (state.get('step') == 'start' and FILE_PATH and os.path.exists(FILE_PATH)):
    FILE_PATH = normalize_audio(FILE_PATH)
    state['downloaded_file'] = FILE_PATH

# 3. Cấu hình tiêu đề mặc định
video_title = os.path.splitext(os.path.basename(FILE_PATH))[0]
if video_title.startswith('downloaded_video_'):
    # try to keep a clean title if it's yt-dlp temp name
    video_title = "Video"
video_description = MO_TA_VIDEO
video_tags = []




# --- XỬ LÝ CẮT VIDEO (NẾU CÓ) ---
if state.get('step') == 'downloaded':
    upload_files = []
    if CHIA_NHO_VIDEO != "Không cắt":
        num_parts = int(CHIA_NHO_VIDEO.split(' ')[2])
        print(f"\n✂️ Đang chia nhỏ video làm {num_parts} phần để tránh bản quyền...")
        
        import subprocess
        import glob
        
        duration_cmd = f'ffprobe -v error -show_entries format=duration -of default=noprint_wrappers=1:nokey=1 "{FILE_PATH}"'
        try:
            total_duration = float(subprocess.check_output(duration_cmd, shell=True).decode('utf-8').strip())
            segment_duration = total_duration / num_parts
            
            base_name = os.path.splitext(FILE_PATH)[0]
            ext = os.path.splitext(FILE_PATH)[1]
            out_pattern = f"{base_name}_Phan_%03d{ext}"
            
            split_cmd = f'ffmpeg -i "{FILE_PATH}" -c copy -f segment -segment_time {segment_duration} -reset_timestamps 1 "{out_pattern}" -y -v warning'
            subprocess.run(split_cmd, shell=True, check=True)
            
            split_files = sorted(glob.glob(f"{base_name}_Phan_*{ext}"))
            if len(split_files) > 0:
                for i, sf in enumerate(split_files):
                    upload_files.append({
                        "path": sf,
                        "title": f"{video_title} - Phần {i+1}",
                        "uploaded": False
                    })
                print(f"✅ Đã cắt xong thành {len(upload_files)} phần!")
            else:
                upload_files.append({"path": FILE_PATH, "title": video_title, "uploaded": False})
        except Exception as e:
            print(f"⚠️ Không thể cắt video: {e}, sẽ tải lên toàn bộ video gốc.")
            upload_files.append({"path": FILE_PATH, "title": video_title, "uploaded": False})
    else:
        upload_files.append({"path": FILE_PATH, "title": video_title, "uploaded": False})
    
    state['upload_files'] = upload_files
    state['step'] = 'uploading'
    save_state()
else:
    upload_files = state.get('upload_files', [])
    print(f"✅ Đã có sẵn danh sách {len(upload_files)} phần để upload.")

# 4. Thực hiện upload lên YouTube
# Lọc những file chưa upload
pending_files = [f for f in upload_files if not f.get('uploaded', False)]
print(f"\n🚀 BẮT ĐẦU UPLOAD LÊN YOUTUBE ({len(pending_files)}/{len(upload_files)} Video còn lại)...")

if not pending_files:
    print("✅ Tất cả các phần đã được upload thành công trước đó!")
else:
    youtube = build('youtube', 'v3', credentials=youtube_credentials)

    privacy_map = {
        "Công khai (public)": "public",
        "Riêng tư (private)": "private",
        "Không công khai (unlisted)": "unlisted"
    }
    privacy_status = privacy_map.get(TRANG_THAI_VIDEO, "private")

    import mimetypes

    for u_file in upload_files:
        if u_file.get('uploaded', False):
            continue
            
        f_path = u_file["path"]
        f_title = u_file["title"]
        
        print(f"\n▶️ Đang tải lên: {f_title}")
        
        if not os.path.exists(f_path):
            print(f"⚠️ Không tìm thấy file {f_path}, bỏ qua...")
            u_file['uploaded'] = True
            save_state()
            continue
            
        body = {
            'snippet': {
                'title': f_title,
                'description': video_description,
                'tags': video_tags,
                'categoryId': '22'
            },
            'status': {
                'privacyStatus': privacy_status
            }
        }

        mimetype, _ = mimetypes.guess_type(f_path)
        if not mimetype or not mimetype.startswith('video/'):
            mimetype = 'application/octet-stream'
        media = MediaFileUpload(f_path, mimetype=mimetype, chunksize=1024*1024*100, resumable=True)

        request = youtube.videos().insert(
            part=','.join(body.keys()),
            body=body,
            media_body=media
        )

        response = None
        start_time = time.time()
        try:
            while response is None:
                status, response = request.next_chunk()
                if status:
                    progress = status.progress()
                    elapsed = time.time() - start_time
                    if progress > 0 and elapsed > 0:
                        speed = status.resumable_progress / elapsed / (1024 * 1024)
                        eta = (elapsed / progress) - elapsed
                        print(f"\r⬆️ YT: {int(progress * 100)}% | Tốc độ: {speed:.1f} MB/s | Còn lại: {format_time(eta)}      ", end="")
            
            print(f"\n✅ Hoàn tất tải lên: {f_title}")
            print(f"🔗 Link: https://youtu.be/{response['id']}")
            
            # Đánh dấu đã upload thành công và lưu lại state
            u_file['uploaded'] = True
            save_state()
            
            # Xoá file đã upload xong để giải phóng bộ nhớ
            try: os.remove(f_path)
            except: pass
            
        except (HttpError, ResumableUploadError) as e:
            err_msg = str(e)
            if hasattr(e, 'content'):
                try:
                    err_json = json.loads(e.content.decode('utf-8'))
                    err_msg = err_json.get('error', {}).get('message', err_msg)
                except:
                    try: err_msg = e.content.decode('utf-8')
                    except: pass
            
            print(f"\n❌ LỖI YOUTUBE API khi tải lên {f_title}: {err_msg}")
            if hasattr(e, 'resp') and e.resp.status == 403:
                if "quotaExceeded" in err_msg or "limit" in err_msg.lower():
                    print("👉 NGUYÊN NHÂN: Tài khoản đã hết hạn ngạch (Quota) upload hôm nay. Hãy chạy lại vào ngày mai để tiếp tục up phần này.")
                else:
                    if os.path.exists(token_file):
                        os.remove(token_file)
                    print("👉 NGUYÊN NHÂN: Token xác thực lỗi. Vui lòng chạy lại để đăng nhập lại.")
            elif hasattr(e, 'resp') and e.resp.status == 400:
                if "duplicate" in err_msg.lower():
                    print("👉 NGUYÊN NHÂN: YouTube phát hiện video này bị trùng lặp.")
            
            raise e

# Nếu tất cả đã hoàn thành, xoá file gốc và xoá file state
all_done = all(f.get('uploaded', False) for f in upload_files)
if all_done:
    if CHIA_NHO_VIDEO != "Không cắt" and os.path.exists(FILE_PATH):
        try: os.remove(FILE_PATH)
        except: pass
    if os.path.exists(STATE_FILE):
        try: os.remove(STATE_FILE)
        except: pass
    print(f"\n🎉 TẤT CẢ QUÁ TRÌNH HOÀN TẤT VÀ DỌN DẸP SẠCH SẼ!")
